In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv


load_dotenv()

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
dbname = os.getenv('DB_NAME')

engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}")

In [2]:
fact_pump_prices = pd.read_sql("SELECT * FROM fact_pump_prices", engine)
fact_excise_rates = pd.read_sql("SELECT * FROM fact_excise_rates", engine)
fact_pump_prices.head()

,week_date,petrol_price,diesel_price
0,2026-08-03,1817.3,1845.6
1,2026-07-27,1757.7,1751.7
2,2026-07-20,1720.2,1695.0
3,2026-07-13,1712.5,1689.3
4,2026-07-06,1729.8,1712.7


In [3]:
fact_excise_rates.head()

,country,effective_date,petrol_excise,diesel_excise,heating_gas_oil,fuel_oil,fuel_oil_high_sulphur,lpg
0,IE_,2026-03-25,584.18,453.15,193.06,NaN,NaN,None
1,IE_,2024-10-09,688.78,595.68,199.17,NaN,NaN,None
2,IE_,2024-08-01,671.43,575.61,199.17,NaN,NaN,None
3,IE_,2024-05-01,NaN,NaN,184.30,NaN,NaN,None
4,IE_,2024-04-01,638.91,551.22,163.96,197.64,NaN,None


In [4]:
print(fact_pump_prices.shape)
print(fact_excise_rates.shape)

(1078, 3)
(38, 8)


In [5]:
fact_excise_rates['effective_date'].sort_values().diff().min()

Timedelta('22 days 00:00:00')

In [6]:
fact_excise_rates['effective_date'].sort_values().diff().describe()

count                          37
mean     378 days 05:11:21.081081
std      413 days 15:54:12.675969
min              22 days 00:00:00
25%             143 days 00:00:00
50%             205 days 00:00:00
75%             491 days 00:00:00
max            1777 days 00:00:00
Name: effective_date, dtype: object

In [7]:
fact_excise_rates.sort_values('effective_date')[['effective_date', 'petrol_excise', 'diesel_excise']]

,effective_date,petrol_excise,diesel_excise
37,1987-12-01,NaN,NaN
36,1988-02-01,NaN,223.100000
35,1990-05-01,277.9,NaN
34,1992-05-01,261.4,NaN
33,1993-02-24,NaN,NaN
32,1994-01-26,273.8,235.500000
31,1995-06-01,277.55,239.250000
30,1996-01-24,285.8,247.500000
29,1997-01-22,298.25,259.950000
28,2000-12-07,274.44,196.140000


In [8]:
fact_excise_rates = fact_excise_rates[['effective_date', 'petrol_excise', 'diesel_excise']]

In [10]:
fact_pump_prices_sorted = fact_pump_prices.sort_values('week_date')

excise_petrol = fact_excise_rates.dropna(subset=['petrol_excise']).sort_values('effective_date')
excise_diesel = fact_excise_rates.dropna(subset=['diesel_excise']).sort_values('effective_date')

event_weeks_petrol = pd.merge_asof(
    excise_petrol[['effective_date', 'petrol_excise']],
    fact_pump_prices_sorted[['week_date']],
    left_on='effective_date',
    right_on='week_date',
    direction='backward'
)

event_weeks_diesel = pd.merge_asof(
    excise_diesel[['effective_date', 'diesel_excise']],
    fact_pump_prices_sorted[['week_date']],
    left_on='effective_date',
    right_on='week_date',
    direction='backward'
)

event_weeks_petrol.shape, event_weeks_diesel.shape

((26, 3), (25, 3))

In [11]:
event_weeks_petrol

,effective_date,petrol_excise,week_date
0,1990-05-01,277.9,NaT
1,1992-05-01,261.4,NaT
2,1994-01-26,273.8,NaT
3,1995-06-01,277.55,NaT
4,1996-01-24,285.8,NaT
5,1997-01-22,298.25,NaT
6,2000-12-07,274.44,NaT
7,2002-01-01,401.36,NaT
8,2002-12-04,0,NaT
9,2003-12-04,442.679999999997,NaT


In [12]:
event_weeks_diesel

,effective_date,diesel_excise,week_date
0,1988-02-01,223.100000,NaT
1,1994-01-26,235.500000,NaT
2,1995-06-01,239.250000,NaT
3,1996-01-24,247.500000,NaT
4,1997-01-22,259.950000,NaT
5,2000-12-07,196.140000,NaT
6,2002-01-01,301.940000,NaT
7,2002-12-04,326.740000,NaT
8,2003-12-04,368.060000,NaT
9,2009-04-07,401.847059,2009-04-06


In [13]:
event_weeks_petrol = event_weeks_petrol.dropna(subset=['week_date'])
event_weeks_diesel = event_weeks_diesel.dropna(subset=['week_date'])

event_weeks_petrol.shape, event_weeks_diesel.shape

((16, 3), (16, 3))

In [15]:
fact_pump_prices_sorted = fact_pump_prices_sorted.reset_index(drop=True)
fact_pump_prices_sorted

,week_date,petrol_price,diesel_price
0,2005-01-03,1009.0,975.0
1,2005-01-10,1009.0,975.0
2,2005-01-17,941.0,949.0
3,2005-01-24,941.0,949.0
4,2005-01-31,941.0,949.0
...,...,...,...
1073,2026-07-06,1729.8,1712.7
1074,2026-07-13,1712.5,1689.3
1075,2026-07-20,1720.2,1695.0
1076,2026-07-27,1757.7,1751.7


In [16]:
idx = fact_pump_prices_sorted.index[fact_pump_prices_sorted['week_date'] == '2008-10-13'][0]
idx

186

In [17]:
window = fact_pump_prices_sorted.iloc[idx-4 : idx+5]
window

,week_date,petrol_price,diesel_price
182,2008-09-15,1329.0,1429.0
183,2008-09-22,1329.0,1429.0
184,2008-09-29,1276.0,1362.0
185,2008-10-06,1276.0,1362.0
186,2008-10-13,1276.0,1362.0
187,2008-10-20,1258.0,1318.0
188,2008-10-27,1258.0,1318.0
189,2008-11-03,1258.0,1318.0
190,2008-11-10,1258.0,1318.0


In [18]:
window = window.copy()
window['relative_week'] = range(-4, 5)
window

,week_date,petrol_price,diesel_price,relative_week
182,2008-09-15,1329.0,1429.0,-4
183,2008-09-22,1329.0,1429.0,-3
184,2008-09-29,1276.0,1362.0,-2
185,2008-10-06,1276.0,1362.0,-1
186,2008-10-13,1276.0,1362.0,0
187,2008-10-20,1258.0,1318.0,1
188,2008-10-27,1258.0,1318.0,2
189,2008-11-03,1258.0,1318.0,3
190,2008-11-10,1258.0,1318.0,4


In [19]:
price_at_event = window.loc[window['relative_week'] == 0, 'petrol_price'].values[0]
price_at_event

np.float64(1276.0)

In [20]:
window['normalized_price'] = (window['petrol_price'] - price_at_event) / price_at_event
window

,week_date,petrol_price,diesel_price,relative_week,normalized_price
182,2008-09-15,1329.0,1429.0,-4,0.041536
183,2008-09-22,1329.0,1429.0,-3,0.041536
184,2008-09-29,1276.0,1362.0,-2,0.000000
185,2008-10-06,1276.0,1362.0,-1,0.000000
186,2008-10-13,1276.0,1362.0,0,0.000000
187,2008-10-20,1258.0,1318.0,1,-0.014107
188,2008-10-27,1258.0,1318.0,2,-0.014107
189,2008-11-03,1258.0,1318.0,3,-0.014107
190,2008-11-10,1258.0,1318.0,4,-0.014107


In [23]:
def get_event_window(event_week_date, price_col, window_size=4):
    idx = fact_pump_prices_sorted.index[fact_pump_prices_sorted['week_date'] == event_week_date][0]
    window = fact_pump_prices_sorted.iloc[idx - window_size : idx + window_size + 1].copy()
    window['relative_week'] = range(-window_size, window_size + 1)
    price_at_event = window.loc[window['relative_week'] == 0, price_col].values[0]
    window['normalized_price'] = (window[price_col] - price_at_event) / price_at_event
    window['event_date'] = event_week_date
    return window

# petrol
petrol_windows = []
for _, row in event_weeks_petrol.iterrows():
    w = get_event_window(row['week_date'], 'petrol_price')
    petrol_windows.append(w)

petrol_all_events = pd.concat(petrol_windows, ignore_index=True)

# diesel
diesel_windows = []
for _, row in event_weeks_diesel.iterrows():
    w = get_event_window(row['week_date'], 'diesel_price')
    diesel_windows.append(w)

diesel_all_events = pd.concat(diesel_windows, ignore_index=True)

# average profile across all events
petrol_profile = petrol_all_events.groupby('relative_week')['normalized_price'].mean()
diesel_profile = diesel_all_events.groupby('relative_week')['normalized_price'].mean()

print("Petrol average profile (% change from event week):")
print(petrol_profile)
print("\nDiesel average profile (% change from event week):")
print(diesel_profile)

Petrol average profile (% change from event week):
relative_week
-4   -0.007991
-3   -0.004993
-2   -0.010009
-1   -0.006050
 0    0.000000
 1    0.007946
 2    0.007979
 3    0.013085
 4    0.015579
Name: normalized_price, dtype: float64

Diesel average profile (% change from event week):
relative_week
-4   -0.027065
-3   -0.022561
-2   -0.020676
-1   -0.013073
 0    0.000000
 1    0.014918
 2    0.014713
 3    0.021443
 4    0.026885
Name: normalized_price, dtype: float64


## Interpretation

Average price behavior around excise duty changes (16 petrol events,
16 diesel events, ±4 week window, normalized to price in the event week):

- **Diesel**: price rises steadily from -2.7% (4 weeks before) to +2.7%
  (4 weeks after) — a ~5.4pp swing around the typical excise change
- **Petrol**: same direction, weaker — from -0.8% to +1.6%, a ~2.3pp swing

Diesel prices appear roughly twice as sensitive to excise changes as
petrol. Note this shows correlation around the event window, not isolated
causation — excise changes may cluster with broader oil price movements,
so this reflects the general market response around these dates rather
than a pure tax pass-through effect.

Both fuels show a gradual, not sharp, price shift — prices trend upward
in the weeks leading into an excise change and continue rising afterward,
rather than jumping discretely at the effective date.

In [24]:
petrol_all_events['fuel_type'] = 'petrol'
diesel_all_events['fuel_type'] = 'diesel'

excise_events_combined = pd.concat([petrol_all_events, diesel_all_events], ignore_index=True)
excise_events_combined.to_parquet('../data/processed/excise_event_study_petrol_diesel.parquet', index=False)

In [25]:
check = pd.read_parquet('../data/processed/excise_event_study_petrol_diesel.parquet')
check.columns

Index(['week_date', 'petrol_price', 'diesel_price', 'relative_week',
       'normalized_price', 'event_date', 'fuel_type'],
      dtype='str')